# 🚀 NanoGPT Lab — Super-AI Cloud GPU Training Notebook
### Fine-Tune Modern 1B - 8B Frontier Models (DeepSeek, LLaMA-3, Qwen) on Free Google Colab GPUs

This notebook allows you to scale beyond laptop CPU training. Using Google Colab's free **NVIDIA Tesla T4 GPU**, you can fine-tune high-intelligence models on datasets created from ChatGPT, Claude, and DeepSeek.

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Install high-speed fine-tuning packages (Unsloth + PyTorch)
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Step 3: Load Frontier Model with 4-bit Quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect Float16 / Bfloat16
load_in_4bit = True # 4x memory savings, fits easily on free Colab GPU

# Choose model: Llama-3.2 (1B/3B) or DeepSeek-R1-Distill-Qwen (1.5B/7B)
model_name = "unsloth/Llama-3.2-1B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Model loaded successfully on GPU!")

In [ ]:
# Step 4: Configure LoRA (Low-Rank Adaptation) for Super-Fast Training
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("LoRA Adapters attached successfully!")

In [ ]:
# Step 5: Format Synthetic Q&A Dataset (ChatGPT / Claude / DeepSeek Distillation)
from datasets import Dataset

# Sample instruction-response pairs
data = {
    "instruction": [
        "What is Python?",
        "Write a Python function to check prime numbers.",
        "Explain quantum computing in simple terms."
    ],
    "response": [
        "Python is a high-level interpreted programming language renowned for its simplicity, readability, and rich ecosystem of AI libraries.",
        "def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True",
        "Quantum computing uses quantum mechanics principles, such as superposition and entanglement, allowing qubits to evaluate multiple probabilities simultaneously."
    ]
}

dataset = Dataset.from_dict(data)
print("Dataset prepared with", len(dataset), "conversations")

In [ ]:
# Step 6: Train the Model
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "response",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

trainer.train()
print("Training complete!")

In [ ]:
# Step 7: Export to GGUF format to run locally on your laptop without GPU!
model.save_pretrained_gguf("my_super_ai_q4_k_m", tokenizer, quantization_method = "q4_k_m")
print("GGUF model exported! You can download this file and run it locally on your laptop!")